# Pricing Challenge: GPT vs Jev

We're comparing two very different ways of getting an AI to estimate a product's price:

Frontier LLMs (GPT-4.1 Nano, GPT-5.6 Luna) — generate a price as text, based purely on their world knowledge. No training, no fine-tuning.

Jev (TypeSafe) — doesn't generate text at all. It returns a probability distribution over a fixed set of price buckets, and we do the arithmetic ourselves to turn that into a number.

All approaches are given the same 200 Amazon products and scored the same way: average error, MSE, r², cost, and latency.

Along the way, we also show that how you read a probability distribution matters almost as much as the distribution itself — a small refinement to how `Jev`'s answer is processed turns it from the weakest performer into the best one, with no retraining involved.

In [ ]:
# Import dependencies

import os
import numpy as np

from dotenv import load_dotenv
from huggingface_hub import login
from litellm import completion

from items import Item
from evaluator import evaluate

In [31]:
# Load environment variables
load_dotenv(override=True)

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

hf_dataset_name = os.environ['HF_DATASET']

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [30]:
train, val, test = Item.from_hub(hf_dataset_name)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


## Frontier models - GPT Nano & Luna

Let's see how frontier models do out of the box; no training, just inference based on their world knowledge.

In [5]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [6]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [32]:
# Using GPT 4.1 Nano via OpenRouter
def nano(item):
    response = completion(model="openrouter/openai/gpt-4.1-nano", messages=messages_for(item), seed=42)
    return response.choices[0].message.content, response._hidden_params["response_cost"]



In [33]:
nano(test[0])[0]

'$250'

In [34]:
test[0].price

219.0

In [14]:
evaluate(nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $34 $25 $10 $20 $80 $6 $65 $1 $870 $363 $71 $25 $9 $34 $8 $71 $5 $90 $31 $74 $76 $35 $124 $132 $303 $355 $5 $301 $64 $30 $60 $10 $40 $35 $219 $90 $21 $36 $23 $180 $45 $23 $105 $70 $0 $22 $13 $65 $52 $20 $95 $125 $10 $197 $16 $8 $25 $52 $13 $86 $28 $21 $10 $179 $30 $90 $295 $25 $44 $16 $8 $30 $1 $0 $21 $26 $3 $3 $7 $30 $4 $15 $74 $11 $0 $68 $44 $60 $24 $13 $20 $5 $10 $4 $78 $1 $7 $20 $425 $50 $3 $11 $11 $150 $32 $9 $370 $6 $51 $20 $136 $59 $48 $54 $40 $5 $4 $24 $47 $34 $311 $80 $46 $0 $15 $20 $51 $90 $64 $34 $13 $65 $5 $85 $10 $55 $10 $78 $62 $11 $0 $70 $11 $74 $118 $15 $90 $65 $18 $1 $144 $7 $10 $3 $29 $101 $41 $20 $5 $410 $19 $11 $2 $60 $7 $752 $30 $5 $5 $10 $3 $30 $3 $27 $101 $3 $57 $19 $18 $546 $15 $150 $1 $100 $13 $73 $7 $25 $12 $15 $99 $15 $11 $41 $119 $10 $70 $26 $1 
Total API cost $0.0024 for 200 items, 36.4s wall clock with 5 workers


In [15]:
def luna(item):
    response = completion(model="openrouter/openai/gpt-5.6-luna", messages=messages_for(item), seed=42, reasoning_effort="none")
    return response.choices[0].message.content, response._hidden_params["response_cost"]
evaluate(luna, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $64 $25 $30 $20 $70 $64 $95 $11 $10 $37 $21 $5 $9 $44 $3 $11 $10 $90 $79 $16 $39 $5 $75 $82 $254 $245 $5 $81 $64 $15 $20 $30 $60 $35 $319 $35 $41 $64 $13 $174 $40 $10 $15 $90 $5 $1 $8 $75 $22 $26 $105 $225 $0 $27 $46 $6 $90 $22 $5 $106 $38 $46 $10 $29 $9 $20 $295 $30 $14 $16 $3 $20 $0 $20 $17 $24 $0 $8 $6 $30 $1 $5 $74 $7 $25 $2 $126 $30 $21 $8 $15 $0 $15 $3 $48 $4 $27 $90 $125 $30 $33 $7 $40 $29 $32 $15 $355 $6 $119 $20 $106 $4 $28 $54 $80 $10 $0 $34 $347 $14 $61 $5 $36 $20 $20 $5 $21 $9 $59 $204 $17 $0 $0 $185 $1 $45 $10 $22 $12 $16 $349 $40 $7 $74 $8 $10 $60 $85 $8 $3 $74 $17 $10 $6 $99 $71 $36 $45 $10 $10 $17 $11 $2 $590 $7 $851 $20 $0 $3 $0 $3 $170 $2 $46 $1 $2 $27 $14 $8 $45 $15 $200 $69 $10 $8 $53 $21 $30 $1 $5 $6 $5 $111 $20 $50 $10 $60 $26 $11 
Total API cost $0.0059 for 200 items, 44.8s wall clock with 5 workers


# And now - Jev

`Jev` is a System One model from TypeSafe: it doesn't generate text, it returns a probability distribution over options that we define up front.

There's no continuous number to ask for, so we split the price range into $20 wide buckets and ask one `Choice` question. Jev picks a bucket and hands back a probability for every one of them; we do the arithmetic ourselves in code.

It's served through OpenRouter's Decisions API, which is a different endpoint to chat completions, so litellm isn't involved here.

In [16]:
from openrouter import OpenRouter

BUCKET_WIDTH = 20
TOP_PRICE = 1000

buckets = [(low, low + BUCKET_WIDTH) for low in range(0, TOP_PRICE, BUCKET_WIDTH)]
criteria = {f"${low}-${high}": f"The product sells for between ${low} and ${high}" for low, high in buckets}
midpoints = {f"${low}-${high}": (low + high) / 2 for low, high in buckets}

# The Decisions API lives outside the usual /api/v1 prefix, hence the server_url
openrouter = OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"], server_url="https://openrouter.ai")

print(f"{len(criteria)} buckets, from {buckets[0]} to {buckets[-1]}")

50 buckets, from (0, 20) to (980, 1000)


In [17]:
questions={
            "price": {
                "type": "choice",
                "instructions": "How much does this product cost at retail in US dollars?",
                "criteria": criteria,
            }
        }

In [18]:
def jev(item):
    response = openrouter.alpha.decisions.create(model="typesafe/jev-1.13", state=item.summary, questions=questions)
    answer = response.answers["price"]
    return midpoints[answer.choice], response.usage.cost

In [19]:
jev(test[0])[0]

210.0

In [20]:
test[0].price

219.0

In [21]:
evaluate(jev, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$9 $94 $25 $40 $0 $200 $114 $105 $6 $80 $503 $69 $10 $24 $49 $3 $31 $20 $0 $89 $104 $86 $5 $35 $122 $243 $85 $10 $111 $50 $10 $30 $70 $50 $25 $309 $30 $26 $4 $8 $170 $50 $5 $165 $110 $5 $12 $2 $55 $8 $5 $100 $265 $30 $157 $44 $3 $50 $8 $18 $26 $48 $61 $20 $219 $0 $70 $335 $85 $134 $2 $13 $70 $11 $20 $6 $86 $0 $13 $9 $90 $2 $0 $69 $3 $10 $28 $96 $30 $16 $8 $5 $10 $5 $2 $138 $4 $67 $60 $365 $10 $67 $7 $49 $159 $92 $15 $370 $1 $211 $30 $104 $59 $38 $14 $40 $5 $10 $154 $107 $14 $121 $20 $76 $30 $30 $10 $51 $31 $49 $19 $73 $5 $0 $25 $5 $115 $20 $98 $72 $26 $40 $30 $10 $104 $58 $15 $300 $55 $3 $9 $104 $17 $50 $11 $69 $61 $26 $110 $20 $79 $16 $48 $3 $150 $7 $542 $20 $10 $20 $5 $8 $210 $7 $52 $39 $2 $117 $4 $63 $336 $5 $90 $39 $60 $8 $43 $37 $30 $3 $55 $39 $5 $71 $70 $10 $40 $70 $21 $4 
Total API cost $0.0165 for 200 items, 12.6s wall clock with 5 workers


## Reading the distribution better

We will making 2 changes here. Neither one teaches `Jev` anything new about prices, we're just asking it better and reading its answer better.

### Change 1: Ask the same question twice, at two different levels of detail.

We ask `Jev` to bucket the price two ways at once (still just one API call): once with narrow $20 buckets, and once with wide $100 buckets. Then we line both answers up on the same $20 grid (splitting each wide bucket's probability evenly across the five narrow buckets inside it) and multiply the two together. A bucket only stays "likely" in the combined result if both the narrow and wide answers agreed it was likely. Wherever they disagree, that disagreement cancels itself out.

### Change 2: Use the median to read the result, not the "top pick" or the mean.

The usual way to read a probability distribution is to either take the single most likely option (the `argmax`) or take a probability-weighted average (the mean). Both can be misleading here:

The top pick can be misleading. When `Jev` is uncertain, its guesses get spread thinly across many buckets. The "winning" bucket might only hold 15% of the total probability, barely edging out its neighbors, not because it's clearly the right answer, but because it happened to be marginally ahead. The median doesn't have this problem: for a bucket to be the median, real probability actually has to be piled up around it.

The weighted mean can be misleading too, in the opposite direction. `Jev`'s probabilities get rounded, so even buckets it considers essentially impossible still show up as a small nonzero number (0.01). With 50 buckets, there can be a lot of these "leftover" buckets scattered far from the real answer. Individually tiny, but added together they pull the average off in their direction. The median isn't affected by this, because it only cares about where the middle of the distribution sits, not how far a thin tail stretches.

Read [here](1pass-jev-problem.md) for further details about these problems.

In [ ]:
# Define width of coarse buckets, buckets of wider width
COARSE_WIDTH = 100

# Define the numerical boundaries of coarse buckets and their criteria
coarse_buckets = [(low, low + COARSE_WIDTH) for low in range(0, TOP_PRICE, COARSE_WIDTH)]
coarse_criteria = {f"${low}-${high}": f"The product sells for between ${low} and ${high}" for low, high in coarse_buckets}

print(coarse_buckets)
print(coarse_criteria)

[(0, 100), (100, 200), (200, 300), (300, 400), (400, 500), (500, 600), (600, 700), (700, 800), (800, 900), (900, 1000)]
{'$0-$100': 'The product sells for between $0 and $100', '$100-$200': 'The product sells for between $100 and $200', '$200-$300': 'The product sells for between $200 and $300', '$300-$400': 'The product sells for between $300 and $400', '$400-$500': 'The product sells for between $400 and $500', '$500-$600': 'The product sells for between $500 and $600', '$600-$700': 'The product sells for between $600 and $700', '$700-$800': 'The product sells for between $700 and $800', '$800-$900': 'The product sells for between $800 and $900', '$900-$1000': 'The product sells for between $900 and $1000'}


In [ ]:
bucket_names = list(criteria)           # Fine bucket labels
coarse_names = list(coarse_criteria)    # Coarse bucket labels

# Lower boundries of fine buckets
lows = np.array([low for low, high in buckets], dtype=float)

PRICE_QUESTION = "How much does this product cost at retail in US dollars?"

# print(bucket_names)
# print(coarse_names)
# print(lows)

In [ ]:
# Takes a resolution's answer (either the fine $20-bucket answer or the coarse $100-bucket answer) and lays it out 
# as an array aligned to the $20 grid.
def spread(probabilities, names, width):
    """Spread one grid's probabilities evenly across the $20 buckets each of them covers."""
    per_bucket = width // BUCKET_WIDTH
    values = np.array([probabilities.get(name, 0.0) for name in names])
    return np.repeat(values / per_bucket, per_bucket)

```python
values = np.array([probabilities.get(name, 0.0) for name in names])
```

`probabilities` is the dict `Jev` returns for the price question, something like {"$200-$300": 0.4, "$300-$400": 0.1, ...}. `names` is the full ordered list of bucket labels for this resolution (`bucket_names` for fine, `coarse_names` for coarse).

The list comprehension walks through every bucket name and looks up its probability with `.get(name, 0.0)`. The 0.0 default matters: if Jev's response happens to omit a bucket (because it rounded that bucket's probability down to nothing, or it just isn't present in the dict), this treats it as zero instead of raising a `KeyError`. The result, `values`, is a plain NumPy array of probabilities, one per bucket, always in the same order as names.

```python
np.repeat(array, per_bucket)
```

Repeats each element of the array per_bucket times, back to back. For example, `np.repeat([0.08, 0.02], 5)` gives:
`[0.08, 0.08, 0.08, 0.08, 0.08, 0.02, 0.02, 0.02, 0.02, 0.02]`

So a single coarse bucket's probability (already divided by 5) gets expanded into five consecutive $20-wide slots, each holding an equal share.

### Put together, with a concrete example

Say the coarse answer says `"$200-$300": 0.4`. That $100-wide bucket covers five $20-wide sub-buckets: `$200-$220, $220-$240, $240-$260, $260-$280, $280-$300`. This code:

- Looks up 0.4 for that bucket name.
- Divides by 5 → 0.08.
- Repeats it 5 times → [0.08, 0.08, 0.08, 0.08, 0.08].

Those five values slot into the corresponding five positions of the $20-wide output array, and the same happens for every other coarse bucket, concatenated together into one long array the same length as the fine grid.

### Why this matters for the rest of the code

This is what lets fine and coarse probabilities be multiplied together element-wise in `jev_two_pass_median` (`combined = fine * coarse`), even though Jev originally answered them on two different bucket widths. The coarse answer is "smeared" evenly across its constituent fine buckets, on the (reasonable) assumption that a $100 bucket's probability is roughly evenly distributed across the five $20 buckets inside it, since the model wasn't asked to distinguish between them at that resolution.





In [ ]:
# Takes a probability distribution over the $20-wide price buckets and returns a single dollar amount: 
# the price at which cumulative probability crosses 50%, interpolated to a precise point rather than just a bucket.
def median_price(distribution):
    """The price at which the distribution reaches half its mass, interpolated inside the bucket."""
    
    # Normalize the distribution
    distribution = distribution / distribution.sum()
    
    # Derive a cumulative distribution
    cumulative = np.cumsum(distribution)
    
    # Returns index of the first bucket where cumulative probability reaches or exceeds 50%
    index = min(int(np.searchsorted(cumulative, 0.5)), len(distribution) - 1)
    
    # Cumulative mass before median bucket
    below = cumulative[index - 1] if index else 0.0
    
    # Zero-probability guard
    if distribution[index] == 0:
        return float(lows[index] + BUCKET_WIDTH / 2)
        
    # Return interpolated median value
    return float(lows[index] + (0.5 - below) / distribution[index] * BUCKET_WIDTH)

`distribution = distribution / distribution.sum()` renormalizes so the array sums to exactly 1. This matters because `combined = fine * coarse` (multiplying two distributions together) doesn't produce something that sums to 1 anymore, so this step restores that property before treating it as a proper probability distribution.

`cumulative = np.cumsum(distribution)` builds a cumulative running total across buckets, from the cheapest to the most expensive. If `distribution = [0.1, 0.3, 0.4, 0.2]`, then `cumulative = [0.1, 0.4, 0.8, 1.0]`. Each entry answers: "what fraction of total probability lies at or below this bucket?"

`np.searchsorted` finds the index where `0.5` would need to be inserted into the sorted array cumulative to keep it sorted. In other words, it finds the position of the first element that is `≥ 0.5`.

### A concrete example

```python
cumulative = np.array([0.1, 0.4, 0.8, 1.0])
np.searchsorted(cumulative, 0.5)
```
This returns 2. Why: 0.5 would need to be inserted between 0.4 (index 1) and 0.8 (index 2) to keep the array sorted, so the insertion point is index 2, which also happens to be the index of the first value that is ≥ 0.5. Bucket index 2 (cumulative value 0.8) is therefore "the bucket where the running total first reaches or passes the halfway point", which is exactly the bucket containing the median.

### Default search side

`searchsorted` has a `side` parameter, default `"left"`, which returns the leftmost valid insertion index. That means if cumulative contains an exact 0.5 at some index, say `cumulative = [0.3, 0.5, 0.9]`, `searchsorted(cumulative, 0.5)` returns 1 (inserting before the existing 0.5, not after it), so it correctly picks that bucket as the one crossing the median, rather than skipping past it.

`min(..., len(distribution) - 1)` serves as a edge case safety guard. If literally every bucket before the last were somehow still under 0.5 (shouldn't happen once normalized, but guards against floating-point edge cases), `searchsorted` could return an out-of-range index (`len(distribution)`); clamping it makes sure it always returns a valid array index.

### The interpolation

`lows[index]` is the dollar value at the start of the median bucket (recall `lows = np.array([low for low, high in buckets])`). The fraction `(0.5 - below) / distribution[index]` asks: "of the probability mass that this bucket holds, what fraction of it is needed to get from below up to exactly 0.5?" Multiplying that fraction by BUCKET_WIDTH (20) converts "fraction of the way through the bucket" into "dollars into the bucket." Adding `lows[index]` gives the final price.

### A worked example

Say the median bucket is $200-$220 (`lows[index] = 200`), cumulative mass before it (below) is 0.46, and this bucket itself holds `distribution[index] = 0.08` of probability.

```python
(0.5 - 0.46) / 0.08 = 0.04 / 0.08 = 0.5   → the 50% mark falls halfway through this bucket
200 + 0.5 × 20 = 210
```

So the median price estimate is $210, not just "$200-$220, call it $210 flat" (the bucket midpoint), but a value derived from exactly where inside the bucket the cumulative mass genuinely crosses 50%. If the mass had been more front-loaded (`below = 0.49`), the crossing point would land closer to $200.25; if more back-loaded (`below = 0.42`), closer to $218.

### Net effect

Rather than snapping to a whole bucket, this returns a continuous dollar figure, precise to wherever inside the relevant bucket the distribution's center of mass actually sits.

In [ ]:
# The predictor function that is passed to evaluate(jev_two_pass_median, test). Given an Item, it 
# returns (price_estimate, cost) using a 2 pass technique of fine and coarse buckets.
def jev_two_pass_median(item):
    """Both resolutions in a single request; combine the two distributions and take the median."""
    response = openrouter.alpha.decisions.create(
        model="typesafe/jev-1.13",
        state=item.summary,
        questions={
            "fine": {"type": "choice", "instructions": PRICE_QUESTION, "criteria": criteria},
            "coarse": {"type": "choice", "instructions": PRICE_QUESTION, "criteria": coarse_criteria},
        },
    )
    answers = response.answers

    # Convert each dict into a NumPy array aligned to the common $20 grid
    fine = spread(answers["fine"].probabilities, bucket_names, BUCKET_WIDTH)
    coarse = spread(answers["coarse"].probabilities, coarse_names, COARSE_WIDTH)

    # Combine the 2 distributions
    combined = fine * coarse
    if combined.sum() == 0:
        combined = fine  # the two grids agreed on nothing, so fall back to the fine one alone

    return median_price(combined), response.usage.cost

Unlike single-pass `jev()`, which sends one "price" question, this sends two questions in the same request: "fine" uses the $20-wide criteria buckets, "coarse" uses the $100-wide coarse_criteria buckets. Both ask the identical `PRICE_QUESTION` about the identical product (`item.summary`). Since it's one request, it costs one call, not two, even though you get two independent answers back.

`response.answers` is keyed by question name, so `answers["fine"]` and `answers["coarse"]` give you each question's result separately. Each has a `.probabilities` dict, mapping `bucket label → probability`.

`spread(...)` (covered earlier) converts each dict into a NumPy array aligned to the common $20 grid: the fine answer maps across almost unchanged (`per_bucket = 1`), while the coarse answer gets each $100 bucket's probability evenly divided and repeated across its five constituent $20 buckets. After this, fine and coarse are two arrays of the same length, directly comparable position-by-position.

### Combining the 2 distributions

Element-wise multiplication: a $20 bucket only keeps meaningful probability in combined if both resolutions assigned it decent probability. If the fine grid favors one price region and the coarse grid favors a totally different one (no overlap), every product ends up near zero, the two resolutions disagree and cancels out each other. The `if combined.sum() == 0` fallback catches the case where multiplying wipes out everything, and just uses the fine distribution alone rather than returning a meaningless all-zero array.

### Converting the distribtion into 1 price

`median_price()` (covered in detail earlier) finds the dollar value where cumulative probability in combined crosses 50%, interpolated within the relevant bucket, rather than snapping to a bucket midpoint or being skewed by a long tail.

### The function as a whole

It follows the same shape as `jev()` and `nano()` and `luna()`: takes an `Item`, makes one API call, returns `(estimate, cost)`. What's different is entirely internal, instead of trusting a single resolution's `argmax`, it cross-validates two resolutions against each other and reads the combined result more carefully via the median. That's the whole implementation of the idea described in "Reading the distribution better".

In [42]:
estimate, cost = jev_two_pass_median(test[0])
print(f"{estimate:.2f} vs actual {test[0].price:.2f}, cost ${cost:.6f}")

192.87 vs actual 219.00, cost $0.000097


In [43]:
evaluate(jev_two_pass_median, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$17 $79 $24 $37 $1 $189 $87 $63 $4 $87 $481 $37 $11 $21 $35 $5 $50 $9 $91 $41 $79 $66 $2 $106 $146 $201 $15 $8 $124 $49 $14 $21 $24 $42 $15 $314 $109 $27 $9 $12 $165 $48 $13 $89 $86 $4 $19 $5 $33 $2 $9 $112 $154 $27 $162 $30 $4 $52 $37 $11 $56 $13 $59 $32 $132 $5 $66 $277 $50 $122 $10 $14 $34 $9 $19 $8 $171 $3 $13 $0 $68 $5 $6 $71 $1 $46 $82 $26 $8 $19 $7 $43 $2 $7 $2 $87 $2 $3 $95 $222 $20 $30 $3 $12 $71 $180 $15 $314 $6 $143 $23 $6 $45 $0 $8 $82 $5 $8 $123 $208 $20 $215 $50 $39 $37 $51 $7 $61 $5 $54 $77 $46 $60 $0 $72 $4 $96 $73 $94 $37 $74 $12 $23 $3 $61 $87 $9 $179 $89 $5 $8 $105 $17 $55 $10 $75 $82 $26 $16 $12 $38 $15 $35 $1 $18 $9 $63 $19 $20 $17 $4 $5 $184 $4 $34 $8 $3 $78 $10 $55 $139 $33 $108 $69 $71 $11 $44 $32 $27 $2 $67 $67 $1 $86 $48 $79 $26 $31 $18 $1 
Total API cost $0.0194 for 200 items, 11.8s wall clock with 5 workers


Click [here](results.md) for a detailed analyis of the results.